# tm_final_33 - Financial Tweet Sentiment Classification
## Text Mining 2025/2026 - NOVA IMS · group_33
### Final Pipeline — Restart & Run All

**Primary model:** `nickmuchi/finbert-tone-finetuned-fintwitter-classification` (FinBERT pre-trained on twitter-financial-news-sentiment — the exact same domain).  
Fine-tuned with **10-fold** stratified CV, 10 epochs, cosine LR schedule, label smoothing, fp16 + GradScaler, best-checkpoint-per-fold.  
**Text fix applied:** `ftfy` mojibake correction + truncation/URL cleanup (23.5% of tweets affected; +0.26pp OOF F1-macro vs raw text at 10-fold).

**Ensemble:** Weighted soft-vote of 8 complementary models (FinBERT variants, RoBERTa-large, DeBERTa-v3). Weights optimised via exhaustive OOF grid search over all 2^8 subsets and stored in `results/tables/ensemble_optimal_result.json`. Pre-computed test probabilities in `results/predictions/prob_test_*.csv`.

| Configuration | OOF F1-macro |
|---|---|
| RoBERTa-large baseline (previous submission) | 0.8873 |
| FinBERT fix-text 10ep, 10-fold (primary) | 0.9082 |
| Ensemble 8 models, uniform weights | 0.9184 |
| **Ensemble 8 models, optimised weights** | **0.9201** |

The ensemble gain over the best individual model is statistically significant (bootstrap 95% CI of the F1 difference: [+0.0066, +0.0172], excludes 0; n=1000 resamples).

**Runtime:** ~2 min with cached probabilities (committed to the repo); ~40 min on a CUDA GPU if the primary-model cache is absent (load-or-train logic in Cell 6). **Instructions:** Kernel → Restart Kernel and Run All Cells. Produces `pred_33.csv`.

In [1]:
# Cell 1: Imports and reproducibility
import os, sys, re, time, gc, json, warnings, tempfile
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import ftfy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                             recall_score, classification_report)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_cosine_schedule_with_warmup)

SEED = 42

def seed_all(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all()
print(f'Seed fixed: {SEED}')
print(f'Python: {sys.version[:20]} | torch {torch.__version__}')
print(f'ftfy {ftfy.__version__}')

Seed fixed: 42
Python: 3.11.9 (tags/v3.11.9 | torch 2.12.0+cu130
ftfy 6.3.1


In [2]:
# Cell 3: Preprocessing — ftfy mojibake fix + truncation cleanup
# The dataset (zeroshot/twitter-financial-news-sentiment) has 23.5% of tweets with
# corrupted UTF-8 characters (e.g. â€™ instead of ') and 16.4% truncated with
# U+FFFD replacement chars. ftfy detects and fixes both automatically.
# Ablation: fix_text +0.61pp OOF F1-macro vs raw text (90.75% vs 90.14%).

_TRUNC_RE = re.compile(r'[�…°�…]+\s*(https?://\S*)?$')
_URL_RE    = re.compile(r'https?://\S+')
_TRAIL_RE  = re.compile(r'[\s\-–:]+$')

def fix_tweet(text: str) -> str:
    """Fix mojibake (ftfy) and remove truncation artefacts + bare URLs."""
    text = ftfy.fix_text(str(text))
    text = _TRUNC_RE.sub('', text)
    text = _URL_RE.sub('', text)
    return _TRAIL_RE.sub('', text).strip()

train = pd.read_csv('data/raw/train.csv')
test  = pd.read_csv('data/raw/test.csv')

texts      = [fix_tweet(t) for t in train['text'].tolist()]
test_texts = [fix_tweet(t) for t in test['text'].tolist()]
y = train['label'].values

print(f'Train: {train.shape} | Test: {test.shape}')
print('Label distribution (0=Bearish, 1=Bullish, 2=Neutral):')
print(train['label'].value_counts().sort_index())
print(f'\nSample before: {train["text"].iloc[1][:90]}')
print(f'Sample after:  {texts[1][:90]}')

Train: (9543, 2) | Test: (2388, 2)
Label distribution (0=Bearish, 1=Bullish, 2=Neutral):
label
0    1442
1    1923
2    6178
Name: count, dtype: int64

Sample before: $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.c
Sample after:  $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean


In [3]:
# Cell 4: Configuration and device selection
TAG          = 'finbert_fintwitter_10ep_fixtext'   # cache identity for load-or-train
MODEL_NAME   = 'nickmuchi/finbert-tone-finetuned-fintwitter-classification'
MAXLEN       = 128
LR           = 5e-6
WEIGHT_DECAY = 0.01
EPOCHS       = 10
N_FOLDS      = 10
WARMUP_RATIO = 0.06
LABEL_SMOOTH = 0.05
GRAD_ACCUM   = 1

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    BATCH, EVAL_BATCH = 16, 32
    AMP_DTYPE = torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    USE_SCALER = True
    print(f'GPU: {torch.cuda.get_device_name(0)} | '
          f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB | '
          f'fp16 + GradScaler')
else:
    BATCH, EVAL_BATCH = 8, 32
    AMP_DTYPE, USE_SCALER = None, False
    torch.set_num_threads(min(12, os.cpu_count()))
    print('CPU fallback')

# Inverse-frequency class weights
counts = np.bincount(y, minlength=3)
class_weights = torch.tensor(len(y) / (3 * counts), dtype=torch.float32)
print(f'class weights: {[round(x,3) for x in class_weights.tolist()]}')

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

GPU: NVIDIA GeForce RTX 5070 | VRAM 12.8 GB | fp16 + GradScaler
class weights: [2.206, 1.654, 0.515]


In [4]:
# Cell 5: Model builder, training loop and inference helpers
# Improvements vs notebook v1: GradScaler (fp16), cosine LR warmup,
# label smoothing, best-checkpoint-per-fold (val F1, not last epoch).

def build_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True,
        dtype=torch.float32,   # fp32 params required for GradScaler
    ).to(DEVICE)


@torch.no_grad()
def predict_proba(model, txts):
    model.eval()
    out = []
    for i in range(0, len(txts), EVAL_BATCH):
        enc = tok(txts[i:i+EVAL_BATCH], padding=True, truncation=True,
                  max_length=MAXLEN, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        if DEVICE == 'cuda' and AMP_DTYPE is not None:
            with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                logits = model(**enc).logits
        else:
            logits = model(**enc).logits
        out.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(out)


def run_fold(tr_texts, tr_labels, va_texts, va_y, all_test_texts, fold):
    """Full fold: train with best-ckpt, return (best_va_proba, best_test_proba)."""
    seed_all(SEED + fold)
    model = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = int(np.ceil(len(tr_texts) / BATCH))
    total_steps = (steps_per_epoch // GRAD_ACCUM) * EPOCHS
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE),
                                  label_smoothing=LABEL_SMOOTH)
    scaler = torch.amp.GradScaler('cuda') if USE_SCALER else None

    best_f1, best_va_proba, best_ckpt_path = -1.0, None, None
    n = len(tr_texts)

    with tempfile.TemporaryDirectory() as tmpdir:
        ckpt = Path(tmpdir) / 'best.pt'
        for epoch in range(EPOCHS):
            model.train()
            order = np.random.permutation(n)
            running, t0 = 0.0, time.time()
            optimizer.zero_grad(set_to_none=True)
            for step, i in enumerate(range(0, n, BATCH)):
                bidx = order[i:i+BATCH]
                bt = [tr_texts[j] for j in bidx]
                bl = torch.tensor([tr_labels[j] for j in bidx], dtype=torch.long, device=DEVICE)
                enc = tok(bt, padding=True, truncation=True, max_length=MAXLEN, return_tensors='pt')
                enc = {k: v.to(DEVICE) for k, v in enc.items()}
                if DEVICE == 'cuda' and AMP_DTYPE is not None:
                    with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
                        loss = loss_fn(model(**enc).logits, bl)
                else:
                    loss = loss_fn(model(**enc).logits, bl)
                loss = loss / GRAD_ACCUM
                if scaler: scaler.scale(loss).backward()
                else:       loss.backward()
                running += float(loss.item()) * GRAD_ACCUM
                if (step+1) % GRAD_ACCUM == 0 or (i+BATCH) >= n:
                    if scaler: scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    if scaler: scaler.step(optimizer); scaler.update()
                    else:      optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
            # validate
            va_proba = predict_proba(model, va_texts)
            ep_f1 = f1_score(va_y, va_proba.argmax(1), average='macro')
            star = ' *** best ***' if ep_f1 > best_f1 else ''
            print(f'    fold {fold+1} epoch {epoch+1}/{EPOCHS} '
                  f'loss={running/steps_per_epoch:.4f} val_f1={ep_f1:.6f} '
                  f'({time.time()-t0:.0f}s){star}')
            if ep_f1 > best_f1:
                best_f1 = ep_f1
                best_va_proba = va_proba.copy()
                torch.save(model.state_dict(), ckpt)
        # reload best checkpoint for test predictions
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        best_test_proba = predict_proba(model, all_test_texts)

    print(f'  fold {fold+1} best val macro-F1={best_f1:.6f}')
    return best_va_proba, best_test_proba, best_f1

print('Helpers defined.')

Helpers defined.


In [5]:
# Cell 6: LOAD-OR-TRAIN — 10-fold CV fine-tuning with result caching
# If cached OOF/test probabilities exist (results/predictions/), load them (~2 s).
# Otherwise run the full 10-fold training (~40 min on GPU) and save the cache.

OOF_PATH  = Path(f'results/predictions/oof_proba_{TAG}.npy')
TEST_PATH = Path(f'results/predictions/test_proba_{TAG}.npy')
CSV_PATH  = Path(f'results/predictions/prob_test_{TAG}.csv')
JSON_PATH = Path(f'results/tables/{TAG}_result.json')

if OOF_PATH.exists() and (TEST_PATH.exists() or CSV_PATH.exists()):
    # ---- cached path -------------------------------------------------------
    oof_proba = np.load(OOF_PATH).astype(np.float32)
    if TEST_PATH.exists():
        test_proba_main = np.load(TEST_PATH).astype(np.float32)
    else:
        test_proba_main = pd.read_csv(CSV_PATH)[['p0','p1','p2']].values.astype(np.float32)
    result_cached = json.loads(JSON_PATH.read_text()) if JSON_PATH.exists() else {}
    fold_f1 = result_cached.get('per_fold_f1', [])
    print(f'[CACHED] {TAG}')
    print(f'  OOF proba  : {oof_proba.shape}  | test proba: {test_proba_main.shape}')
    if fold_f1:
        print(f'  per-fold F1: {[round(x,4) for x in fold_f1]}')
else:
    # ---- training path -----------------------------------------------------
    print(f'[TRAINING] {TAG} — cache not found, training {N_FOLDS}-fold from scratch...')
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof_proba      = np.zeros((len(y), 3), dtype=np.float32)
    test_proba_sum = np.zeros((len(test_texts), 3), dtype=np.float32)
    fold_f1 = []
    t_start = time.time()

    for fold, (tr_idx, va_idx) in enumerate(cv.split(texts, y)):
        print(f'\n=== Fold {fold+1}/{N_FOLDS} ===')
        va_proba, test_proba_fold, best_f1 = run_fold(
            [texts[i] for i in tr_idx], [int(y[i]) for i in tr_idx],
            [texts[i] for i in va_idx], y[va_idx], test_texts, fold)
        oof_proba[va_idx] = va_proba
        test_proba_sum   += test_proba_fold
        fold_f1.append(best_f1)
        gc.collect()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    test_proba_main = test_proba_sum / N_FOLDS
    # save cache so the next Restart & Run All loads instantly
    OOF_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(OOF_PATH, oof_proba)
    np.save(TEST_PATH, test_proba_main)
    pd.DataFrame(test_proba_main, columns=['p0','p1','p2']).to_csv(CSV_PATH, index=False)
    print(f'\nDone in {(time.time()-t_start)/60:.1f} min. Cache saved.')

[CACHED] finbert_fintwitter_10ep_fixtext
  OOF proba  : (9543, 3)  | test proba: (2388, 3)
  per-fold F1: [0.9047, 0.9072, 0.8937, 0.9037, 0.9187, 0.9307, 0.8989, 0.9061, 0.9107, 0.9081]


In [6]:
# Cell 7: Out-of-fold performance of the primary model (honest, leak-free estimate)
oof_pred = oof_proba.argmax(1)
oof_f1   = f1_score(y, oof_pred, average='macro')

fold_info = (f'(per-fold {[round(x,4) for x in fold_f1]}, std {np.std(fold_f1):.4f})'
             if fold_f1 else '(per-fold scores in results/tables JSON)')
print(f'OOF F1-macro : {oof_f1:.4f}  {fold_info}')
print(f'Accuracy     : {accuracy_score(y, oof_pred):.4f}')
print(f'Precision-mac: {precision_score(y, oof_pred, average="macro"):.4f}')
print(f'Recall-macro : {recall_score(y, oof_pred, average="macro"):.4f}')
print()
print(classification_report(y, oof_pred, target_names=['Bearish', 'Bullish', 'Neutral']))

OOF F1-macro : 0.9082  (per-fold [0.9047, 0.9072, 0.8937, 0.9037, 0.9187, 0.9307, 0.8989, 0.9061, 0.9107, 0.9081], std 0.0098)
Accuracy     : 0.9293
Precision-mac: 0.9008
Recall-macro : 0.9163

              precision    recall  f1-score   support

     Bearish       0.84      0.89      0.86      1442
     Bullish       0.90      0.92      0.91      1923
     Neutral       0.96      0.94      0.95      6178

    accuracy                           0.93      9543
   macro avg       0.90      0.92      0.91      9543
weighted avg       0.93      0.93      0.93      9543



In [7]:
# Cell 7b: Weighted soft-vote ensemble
# Weights live in results/tables/ensemble_optimal_result.json — found via exhaustive
# OOF grid search (all 2^8 subsets x weight grid {0.25,0.5,0.75,1.0}).
# The primary model trained above always votes with its JSON weight (default 1.0).
# Pre-computed test probabilities: results/predictions/prob_test_{tag}.csv (committed)
# with fallback to test_proba_{tag}.npy (local cache).

WEIGHTS_PATH = Path('results/tables/ensemble_optimal_result.json')

if WEIGHTS_PATH.exists():
    opt = json.loads(WEIGHTS_PATH.read_text())
    ENSEMBLE_WEIGHTS = dict(opt['models'])
    expected_f1 = opt.get('oof_f1_macro')
else:
    ENSEMBLE_WEIGHTS = {TAG: 1.0}
    expected_f1 = None

# primary model: use weight from JSON (it is part of the optimised configuration)
w_primary = ENSEMBLE_WEIGHTS.pop(TAG, 1.0)
ensemble_sum  = test_proba_main.astype(np.float64) * w_primary
ensemble_wsum = w_primary
loaded = [f'{TAG} (w={w_primary}) [primary]']

for tag, w in ENSEMBLE_WEIGHTS.items():
    proba = None
    csv_p = Path(f'results/predictions/prob_test_{tag}.csv')
    npy_p = Path(f'results/predictions/test_proba_{tag}.npy')
    if csv_p.exists():
        proba = pd.read_csv(csv_p)[['p0','p1','p2']].values.astype(np.float64)
    elif npy_p.exists():
        proba = np.load(npy_p).astype(np.float64)
    if proba is not None and proba.shape == (len(test_texts), 3):
        ensemble_sum  += proba * w
        ensemble_wsum += w
        loaded.append(f'{tag} (w={w})')
    else:
        print(f'  WARN: probabilities not found/invalid for {tag} — skipped')

test_proba_final = (ensemble_sum / ensemble_wsum).astype(np.float32)

print(f'Ensemble: {len(loaded)} models, total weight = {ensemble_wsum:.2f}')
for m in loaded:
    print(f'  + {m}')
print(f'\nPrimary model OOF F1-macro    : {oof_f1:.4f}')
if expected_f1:
    print(f'Ensemble OOF F1-macro (cached): {expected_f1:.4f}')

Ensemble: 8 models, total weight = 6.75
  + finbert_fintwitter_10ep_fixtext (w=1.0) [primary]
  + finbert_fintwitter_7ep (w=1.0)
  + roberta_large_ts_v2 (w=1.0)
  + debertav3_large_6ep (w=1.0)
  + finbert_fintwitter_10ep (w=0.75)
  + finbert_fintwitter (w=0.75)
  + debertav3_large_6ep_fixtext (w=0.75)
  + deberta_base_finance_fixtext (w=0.5)

Primary model OOF F1-macro    : 0.9082
Ensemble OOF F1-macro (cached): 0.9201


In [8]:
# Cell 8: Generate predictions and save pred_33.csv (with full validation)
test_pred  = test_proba_final.argmax(1)
submission = pd.DataFrame({'id': test['id'], 'label': test_pred.astype(int)})
submission.to_csv('pred_33.csv', index=False)
os.makedirs('results/predictions', exist_ok=True)
submission.to_csv('results/predictions/pred_final_ensemble.csv', index=False)

print(f'Predictions saved: pred_33.csv ({len(submission)} rows)')
print('Distribution:')
print(submission['label'].value_counts().sort_index()
      .rename({0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}))
print()

# Robust validation of the deliverable
assert len(submission) == 2388,                        f'Expected 2388, got {len(submission)}'
assert list(submission.columns) == ['id', 'label'],    'Columns must be exactly [id, label]'
assert set(submission['label'].unique()) == {0, 1, 2}, f'Labels must be {{0,1,2}}'
assert submission['label'].isna().sum() == 0,          'NaN labels found!'
assert submission['id'].is_unique,                     'Duplicate ids found!'
assert (submission['label'] >= 0).all() and (submission['label'] <= 2).all(), 'Labels out of range'
min_class_pct = submission['label'].value_counts(normalize=True).min()
assert min_class_pct > 0.01, f'Under-represented class ({min_class_pct:.1%}) — possible model collapse'

print('All assertions PASSED.')
print()
print(submission.head(10).to_string(index=False))

Predictions saved: pred_33.csv (2388 rows)
Distribution:
label
Bearish     385
Bullish     497
Neutral    1506
Name: count, dtype: int64

All assertions PASSED.

 id  label
  0      1
  1      2
  2      2
  3      1
  4      2
  5      1
  6      0
  7      0
  8      2
  9      2
